<a href="https://colab.research.google.com/github/misbahhassan6400/flyrank-ml-internship/blob/main/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb huggingface_hub pandas

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN missing. Colab Secrets mein access ON karo.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

files = con.sql("""
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**/*.parquet')
""").df()

feb_matches = files[
    files["file"].astype(str).str.contains("2026-02", regex=False)
]

FEB = feb_matches.iloc[0]["file"]

print("Setup complete")
print("Using February file:", FEB)

Setup complete
Using February file: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet


In [ ]:
signals = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COALESCE(
        SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE),
        0
    ) AS gsc_impressions,

    COALESCE(
        SUM(gsc_clicks)
        FILTER (WHERE gsc_data_available IS TRUE),
        0
    ) AS gsc_clicks,

    AVG(gsc_avg_position)
        FILTER (
            WHERE gsc_data_available IS TRUE
            AND gsc_avg_position IS NOT NULL
        ) AS gsc_avg_position,

    COALESCE(
        SUM(ga4_sessions)
        FILTER (WHERE ga4_data_available IS TRUE),
        0
    ) AS ga4_sessions,

    COALESCE(
        SUM(ga4_engaged_sessions)
        FILTER (WHERE ga4_data_available IS TRUE),
        0
    ) AS ga4_engaged_sessions

FROM read_parquet('{FEB}')
WHERE month = '2026-02'
GROUP BY client_hash_id, content_hash_id
""").df()

signals["ctr"] = np.where(
    signals["gsc_impressions"] > 0,
    signals["gsc_clicks"] / signals["gsc_impressions"],
    np.nan
)

signals["engagement_rate"] = np.where(
    signals["ga4_sessions"] > 0,
    signals["ga4_engaged_sessions"] / signals["ga4_sessions"],
    np.nan
)

print("Rows:", len(signals))
display(signals.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 321546


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0,0.000000,NaN
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0,0.008186,0.000000
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0,0.000000,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,6.0,1.0,0.001024,0.166667
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,0.0,0.002062,0.000000


## 1. Distributions

I inspected the February distributions before setting thresholds. Impressions, clicks, and sessions are expected to be heavy-tailed, so I use medians and upper quantiles rather than relying only on averages. Missing values are also visible because Search Console and GA4 availability varies by row.


In [ ]:
distribution_summary = signals[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "ctr",
        "engagement_rate"
    ]
].describe(
    percentiles=[0.50, 0.90, 0.99]
).T

display(distribution_summary)

missing_zero_summary = pd.DataFrame({
    "missing_n": signals.isna().sum(),
    "zero_n": [
        (signals[col] == 0).sum()
        if pd.api.types.is_numeric_dtype(signals[col])
        else np.nan
        for col in signals.columns
    ]
})

display(missing_zero_summary)

,count,mean,std,min,50%,90%,99%,max
gsc_impressions,321546.0,560.196432,2844.275201,0.0,0.000000,1059.000000,10219.200000,203401.0
gsc_clicks,321546.0,1.823083,15.257068,0.0,0.000000,2.000000,36.000000,3310.0
gsc_avg_position,153559.0,12.702216,13.977253,0.0,7.935419,29.192854,68.653892,633.0
ga4_sessions,321546.0,1.073753,12.322264,0.0,0.000000,1.000000,24.000000,4038.0
ga4_engaged_sessions,321546.0,0.053967,0.870499,0.0,0.000000,0.000000,1.000000,312.0
ctr,153559.0,0.004964,0.040384,0.0,0.000000,0.006079,0.071429,1.0
engagement_rate,33347.0,0.039482,0.130195,0.0,0.000000,0.111111,1.000000,1.0


,missing_n,zero_n
client_hash_id,0,NaN
content_hash_id,0,NaN
gsc_impressions,0,167987.0
gsc_clicks,0,266456.0
gsc_avg_position,167987,1603.0
ga4_sessions,0,288199.0
ga4_engaged_sessions,0,314649.0
ctr,167987,98469.0
engagement_rate,288199,26450.0


The distributions show that the volume fields have long tails when the 99th percentile is much larger than the median. This means thresholds should be treated as directional decision-support rules, not universal truths.

## 2. Signal test #1 / #2 / #3 (verdict each)

I test three signals using only February observations:

1. Search impressions as a volume signal.
2. Average position compared with CTR.
3. GA4 sessions compared with engagement rate.

Each result gets one verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [ ]:
signals["volume_bucket"] = pd.cut(
    signals["gsc_impressions"],
    bins=[-1, 0, 99, 999, np.inf],
    labels=[
        "NO_IMPRESSIONS",
        "LOW_1_TO_99",
        "MEDIUM_100_TO_999",
        "HIGH_1000_PLUS"
    ],
    include_lowest=True
)

volume_test = (
    signals
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_clicks=("gsc_clicks", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print("Signal 1: volume")
display(volume_test)

Signal 1: volume


,volume_bucket,n,median_impressions,median_clicks,median_ctr
0,NO_IMPRESSIONS,167987,0.0,0.0,NaN
1,LOW_1_TO_99,73237,11.0,0.0,0.000000
2,MEDIUM_100_TO_999,47015,314.0,0.0,0.000000
3,HIGH_1000_PLUS,33307,2529.0,6.0,0.002208


**Signal 1 verdict: MIXED.** Impressions vary across buckets, but the click pattern is not consistently increasing. Volume may still be useful for prioritization, but it is not a reliable standalone signal.

In [ ]:
position_data = signals[
    signals["gsc_avg_position"].notna()
    & signals["ctr"].notna()
].copy()

position_data["position_bucket"] = pd.cut(
    position_data["gsc_avg_position"],
    bins=[-np.inf, 10, 20, 50, np.inf],
    labels=[
        "TOP_10",
        "POSITIONS_11_TO_20",
        "POSITIONS_21_TO_50",
        "POSITION_51_PLUS"
    ]
)

position_test = (
    position_data
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("gsc_avg_position", "median"),
        median_ctr=("ctr", "median"),
        low_ctr_n=("ctr", lambda x: (x < 0.02).sum())
    )
    .reset_index()
)

print("Signal 2: position versus CTR")
display(position_test)

**Signal 2 verdict: MIXED.** Position and CTR do not move consistently across every bucket. The signal may help with review prioritization, but it should not be treated as a guaranteed CTR-fix rule.

In [ ]:
engagement_data = signals[
    signals["ga4_sessions"] > 0
    & signals["engagement_rate"].notna()
].copy()

engagement_data["session_bucket"] = pd.cut(
    engagement_data["ga4_sessions"],
    bins=[0, 9, 99, 999, np.inf],
    labels=[
        "SESSIONS_1_TO_9",
        "SESSIONS_10_TO_99",
        "SESSIONS_100_TO_999",
        "SESSIONS_1000_PLUS"
    ]
)

engagement_test = (
    engagement_data
    .groupby("session_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_sessions=("ga4_sessions", "median"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

print("Signal 3: sessions versus engagement rate")
display(engagement_test)

Signal 3: sessions versus engagement rate


,session_bucket,n,median_sessions,median_engagement_rate
0,SESSIONS_1_TO_9,26142,2.0,0.000000
1,SESSIONS_10_TO_99,6770,21.0,0.037037
2,SESSIONS_100_TO_999,432,143.0,0.050390
3,SESSIONS_1000_PLUS,3,1295.0,0.061004


**Signal 3 verdict: MIXED.** Session volume and engagement rate describe different aspects of performance. The bucket table should be used for context, not as proof that high traffic automatically means high engagement.

## 3. The flag-linked test

The flag-linked signal is CTR versus average position. The assumption behind the CTR-fix logic is that content ranking in the top ten but receiving a low CTR may deserve title or snippet review. I test how many February rows meet that condition and compare them with other position groups.


In [ ]:
position_data = signals[
    signals["gsc_avg_position"].notna()
    & signals["ctr"].notna()
].copy()

flag_test = position_data.copy()

flag_test["ctr_fix_candidate"] = (
    (flag_test["gsc_avg_position"] <= 10)
    & (flag_test["ctr"] < 0.02)
    & (flag_test["gsc_impressions"] >= 100)
)

flag_summary = pd.DataFrame({
    "group": [
        "all usable GSC rows",
        "top-10 rows",
        "top-10 low-CTR candidates",
        "positions 11-20 rows"
    ],
    "n": [
        len(flag_test),
        (flag_test["gsc_avg_position"] <= 10).sum(),
        flag_test["ctr_fix_candidate"].sum(),
        (
            (flag_test["gsc_avg_position"] > 10)
            & (flag_test["gsc_avg_position"] <= 20)
        ).sum()
    ]
})

display(flag_summary)

top10 = flag_test[
    flag_test["gsc_avg_position"] <= 10
]

print(
    "Top-10 low-CTR rate:",
    round((top10["ctr"] < 0.02).mean(), 4)
    if len(top10) else "not measurable"
)

print(
    "Top-10 low-CTR candidates with at least 100 impressions:",
    int(flag_test["ctr_fix_candidate"].sum())
)

,group,n
0,all usable GSC rows,153559
1,top-10 rows,95141
2,top-10 low-CTR candidates,51626
3,positions 11-20 rows,31700


Top-10 low-CTR rate: 0.9679
Top-10 low-CTR candidates with at least 100 impressions: 51626


## 4. What this means in practice

The content team should use impressions, position, and CTR to prioritize review rather than to promise outcomes. Top-ten pages with enough impressions and low CTR are reasonable CTR-fix candidates, while high-impression pages outside the top ten may be directional quick-win candidates. Missing or low-volume rows should be treated cautiously because their signals are less stable.

## Self-check

- Distributions were inspected and heavy tails were noted.
- Three February-only signals were tested.
- Every signal has a visible bucket table with n.
- Every signal has one verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.
- The CTR-versus-position test is explicitly linked to the CTR-fix flag logic.
- No March data, labels, future windows, product flags, client names, or URLs were used.
- The notebook runs top to bottom without errors.
- The notebook is committed under `work/notebooks/w04_signal_audit.ipynb`.